# Foundation TimesFM Decoder Demo
This notebook shows how to load the pretrained checkpoint and run zero-shot forecasting on a synthetic univariate series.

In [ ]:
# Imports
import os, numpy as np, pandas as pd, torch
from src.model.decoder_only import TimesFMDecoder
from src.model.layers import TransformerConfig
from src.utils.io import load_checkpoint, load_norm_stats
from src.infer.forecast import generate_forecast

In [ ]:
# Load config
import yaml
with open('configs/default.yaml','r') as f: cfg = yaml.safe_load(f)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Initialize model and load checkpoint
cfg_m = cfg['model']
mconf = TransformerConfig(d_model=cfg_m['d_model'], n_heads=cfg_m['n_heads'], n_layers=cfg_m['n_layers'], ffn_dim=cfg_m['ffn_dim'], dropout=cfg_m['dropout'], context_len=cfg_m['context_len'])
model = TimesFMDecoder(in_dim=cfg_m['in_dim'], config=mconf).to(device)
state, _, extra = load_checkpoint(os.path.join(cfg['paths']['checkpoints_dir'], 'foundation_decoder.pt'), map_location=device)
if state: model.load_state_dict(state)

In [ ]:
# Create synthetic univariate series and tile to model input dims
N = 800
t = np.arange(N)
v = np.sin(2*np.pi*t/50) + 0.1*np.random.randn(N)
x = v[:,None].astype(np.float32)
# mock mean/std with zeros/ones of in_dim
mean = np.zeros(cfg_m['in_dim'], dtype=np.float32)
std = np.ones(cfg_m['in_dim'], dtype=np.float32)
x_tiled = np.tile(x, (1, cfg_m['in_dim']))

In [ ]:
# Forecast
yhat, lo, hi, latents = generate_forecast(model, x_tiled, horizon=90, context_len=cfg_m['context_len'], device=device)
import matplotlib.pyplot as plt
plt.figure(figsize=(10,4))
plt.plot(np.arange(N), x[:,0], label='history')
plt.plot(np.arange(N, N+len(yhat)), yhat[:,0], label='forecast')
plt.fill_between(np.arange(N, N+len(yhat)), lo[:,0], hi[:,0], color='gray', alpha=0.3, label='CI')
plt.legend(); plt.title('Zero-shot forecast (demo)'); plt.show()